# Track 2 · Stage 4 — Evaluation

## Run **GATE 3 first**, before any generation or training.

`whisper-large-v2` zero-shot on the test set should score **≈ 52.0 MER / 42.9 CBA-HE**.
Inference only — no training, ~3 GB, ~40 min on a T4.

That number is published in the paper, so reproducing it validates decoding,
normalization, word-level language ID, and both metrics end to end. It is **not**
our baseline — it is the calibration of the measuring instrument. A broken metric
makes every downstream result uninterpretable.

Then decode M6/M7/M8 and the `whisper-small` zero-shot baseline they are measured against.

Decoding uses `language=None` (Whisper auto-detect), reproducing WhisperX's "None"
option from the paper. That is the whole point: a model that has not learned
code-switching mis-detects the language and then transliterates or deletes.

In [ ]:
!pip install -q git+https://github.com/BRUH-MAIN/codeswitching.git
!pip install -q -U transformers jiwer
!pip install -q "datasets<4" librosa soundfile   # see 02_train: torchcodec

import os, subprocess, sys
os.environ["HF_HOME"] = "/kaggle/temp/hf"
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login; login(token=HF_TOKEN)

REAL = "RohanRamesh/mucs-he-cs"
OUT  = "/kaggle/working"

def run(*args):
    print(">", " ".join(str(a) for a in args), flush=True)
    subprocess.run([sys.executable, "-m", *args], check=True)

## GATE 3 — calibrate the metric against a published number

In [ ]:
run("csasr.eval.decode", "--model", "openai/whisper-large-v2",
    "--test-hf", REAL, "--test-config", "test", "--hf-token", HF_TOKEN,
    "--out", f"{OUT}/hyp_largev2_zeroshot.jsonl", "--batch-size", "8")

In [ ]:
import json
from datasets import load_dataset
from csasr.manifest import write_jsonl
from csasr.eval.score import score

test = load_dataset(REAL, "test", split="train", token=HF_TOKEN)
write_jsonl(f"{OUT}/refs_test.jsonl",
            [{"utt_id": r["utt_id"], "text": r["text"]} for r in test])

for mode in ("word", "hybrid"):
    res = score(f"{OUT}/refs_test.jsonl", f"{OUT}/hyp_largev2_zeroshot.jsonl", mer_mode=mode)
    print(f"MER mode={mode:<7} -> MER {res['mer']:.1f}   CBA-HE {res['cba_he']:.1f}   CBA-EH {res['cba_eh']:.1f}")

print("\npaper (large-v2 zero-shot): MER 52.0   CBA-HE 42.9   CBA-EH 36.x")
print("Whichever mode lands near 52.0 is the definition the authors used.")

## whisper-small zero-shot — the actual baseline for M6/M7/M8

In [ ]:
run("csasr.eval.decode", "--model", "openai/whisper-small",
    "--test-hf", REAL, "--test-config", "test", "--hf-token", HF_TOKEN,
    "--out", f"{OUT}/hyp_small_zeroshot.jsonl", "--batch-size", "16")

## Decode the fine-tuned models

In [ ]:
for m in ("m6", "m7", "m8"):
    run("csasr.eval.decode", "--model", f"RohanRamesh/whisper-small-cs-{m}",
        "--test-hf", REAL, "--test-config", "test", "--hf-token", HF_TOKEN,
        "--out", f"{OUT}/hyp_{m}.jsonl", "--batch-size", "16")

## Results — reproduce the ordering of Table 2

In [ ]:
systems = [
    ("large-v2 zero-shot", "hyp_largev2_zeroshot.jsonl", 52.0),
    ("small zero-shot",    "hyp_small_zeroshot.jsonl",   None),
    ("M6 (T1, 8h)",        "hyp_m6.jsonl",               48.2),
    ("M7 (T2, 22h)",       "hyp_m7.jsonl",               40.8),
    ("M8 (T2 + mono)",     "hyp_m8.jsonl",               39.2),
]
rows = []
for name, f, paper_mer in systems:
    r = score(f"{OUT}/refs_test.jsonl", f"{OUT}/{f}")
    rows.append((name, r, paper_mer))

print(f"{'system':<20}{'MER':>8}{'paper':>8}{'CBA-HE':>9}{'CBA-EH':>9}")
for name, r, pm in rows:
    p = f"{pm:.1f}" if pm else "-"
    print(f"{name:<20}{r['mer']:>8.1f}{p:>8}{r['cba_he']:>9.1f}{r['cba_eh']:>9.1f}")

mers = [r["mer"] for _, r, _ in rows[2:]]
print("\nM6 > M7 > M8 ordering reproduced:", mers == sorted(mers, reverse=True))

with open(f"{OUT}/table2.json", "w") as fh:
    json.dump([{"system": n, **r} for n, r, _ in rows], fh, indent=2)